In [1]:
import matplotlib.pyplot as plt
%matplotlib qt
import numpy as np
import pickle
import pandas as pd

from pathlib import Path
from src.models.sklearn_models import RandomForestModel, XGBModel
from src.models.keras_models import ConcatenatedModulesModel
from src.visualization.radar_chart import radial_chart


In [2]:


rf = RandomForestModel()
xgb = XGBModel()
dnn = ConcatenatedModulesModel("ConcatenatedDNN")
cnn = ConcatenatedModulesModel("ConcatenatedCNN")
bilstm = ConcatenatedModulesModel("ConcatenatedBiLSTM")

column_names = ['chlid', 'chl_a', 'chl_b', 'chc12', 'fucox', 'hxfcx', 'btfcx', 'diadi',
       'allox', 'diato', 'zeaxa', 'betac', 'perid']

graph_names = ['chlide', 'chla', 'chlb', 'chlc12', 'fuco', "hex", "but", 'diad',
       'allo', 'diato', 'zea', 'caro', 'peri']


rename_models = {
    "ConcatenatedDNN": "DNN",
    "ConcatenatedCNN": "CNN",
    "ConcatenatedBiLSTM": "BiLSTM",
    "rf": "RF",
    "xgb": "XGB",
}

colors = ['#1E5799', '#AAAAAA', '#00A0B0', '#F0AD4E', '#C85217']

markers = ['^', 's', 'p', 'p', 'p']
linestyles = ['-', '--', ':', '-.', ':']

In [3]:
models = [cnn, dnn, bilstm, xgb, rf]
# models = [cnn]

experiment_name = 'OLCI'
experiment_name = '13_wl_final'
# experiment_name = '5_wl_final'

In [4]:
path_experiments =  Path(f'../../experiments/{experiment_name}')
paths_metrics = [p / 'metrics' for p in path_experiments.iterdir()]

In [5]:
# Load metrics:
metrics_test = {model.name: { p.parent.name :pd.read_csv(p/ (model.name + '_test.csv'), index_col=0) for p in paths_metrics} for model in models}
metrics_train = {model.name: { p.parent.name :pd.read_csv(p/ (model.name + '_train.csv'), index_col=0) for p in paths_metrics} for model in models}
        

In [6]:
metrics_test_mean = {model_name:  pd.concat(mets.values()).groupby(level=0).mean() for model_name, mets in metrics_test.items()}
metrics_train_mean = {model_name:  pd.concat(mets.values()).groupby(level=0).mean() for model_name, mets in metrics_train.items()}
metrics_test_mean_mean = pd.DataFrame({model_name: val.mean(axis=1) for model_name, val in metrics_test_mean.items()})
metrics_train_mean_mean = pd.DataFrame({model_name: val.mean(axis=1) for model_name, val in metrics_train_mean.items()})

In [7]:
# select values to plot (R2 here)
r2_mets = pd.DataFrame(ds.loc['R2',:].rename(model_name) for model_name, ds in metrics_test_mean.items())

In [8]:
r2_mets = r2_mets.sort_values(axis=1, by='ConcatenatedCNN', ascending=False)
r2_mets = r2_mets.rename(dict(zip(column_names, graph_names)), axis=1)
r2_mets = r2_mets.rename(rename_models, axis=0)

In [9]:
# pigments = np.array(['chlide_a', 'chla   ', 'chlb      ', 'chlc1+c2              ', 'fucox           ', "19'hxfcx               ", 
#                      "19'btfcx         ", "             diadino", "        allox", "           diatox", "            zeaxan", "              beta_car",
#                      "               peridinin"])




In [10]:
radial_chart(r2_mets.values, r2_mets.columns, r2_mets.index, colors=colors, markers=markers, linestyles=linestyles, title='', save=None)

In [11]:
r2_mets

,chla,chlc12,caro,fuco,diad,peri,chlb,diato,chlide,hex,allo,but,zea
CNN,0.927271,0.924595,0.915100,0.898419,0.882132,0.836779,0.791856,0.770290,0.748999,0.669882,0.649530,0.549060,0.548819
DNN,0.871697,0.883277,0.845381,0.872509,0.848491,0.811355,0.678291,0.690064,0.749485,0.597398,0.566663,0.297228,0.293975
BiLSTM,0.894677,0.909448,0.883202,0.882325,0.882208,0.812700,0.724412,0.743249,0.751519,0.715847,0.610763,0.458672,0.409853
XGB,0.912795,0.920872,0.889833,0.896103,0.860384,0.854783,0.770071,0.743582,0.664891,0.551984,0.555442,0.473469,0.429347
RF,0.898049,0.914259,0.883112,0.901023,0.873083,0.853038,0.739886,0.719772,0.745741,0.620450,0.574591,0.416496,0.460731


In [12]:
#  fine tunning model vs scratch:

experiment_name_cnn = 'OLCI_sat_scratch'
experiment_name_cnn_ast = 'OLCI_sat_ft'

path_experiments_cnn =  Path(f'../../experiments/{experiment_name_cnn}')
paths_metrics_cnn = [p / 'metrics' for p in path_experiments_cnn.iterdir()]

path_experiments_cnn_ast =  Path(f'../../experiments/{experiment_name_cnn_ast}')
paths_metrics_cnn_ast = [p / 'metrics' for p in path_experiments_cnn_ast.iterdir()]

# Load metrics:
metrics_test = {
    r'$CNN_{ft}$': { p.parent.name :pd.read_csv(p/ (cnn.name + '_test.csv'), index_col=0) for p in paths_metrics_cnn_ast},
    r'$CNN_{sat}$': { p.parent.name :pd.read_csv(p/ (cnn.name + '_test.csv'), index_col=0) for p in paths_metrics_cnn},
               }
metrics_train = {r'$CNN_{ft}$': { p.parent.name :pd.read_csv(p/ (cnn.name + '_train.csv'), index_col=0) for p in paths_metrics_cnn_ast},
                 r'$CNN_{sat}$': { p.parent.name :pd.read_csv(p/ (cnn.name + '_train.csv'), index_col=0) for p in paths_metrics_cnn},
               }        

In [13]:
metrics_test_mean = {model_name:  pd.concat(mets.values()).groupby(level=0).mean() for model_name, mets in metrics_test.items()}
metrics_train_mean = {model_name:  pd.concat(mets.values()).groupby(level=0).mean() for model_name, mets in metrics_train.items()}
metrics_test_mean_mean = pd.DataFrame({model_name: val.mean(axis=1) for model_name, val in metrics_test_mean.items()})
metrics_train_mean_mean = pd.DataFrame({model_name: val.mean(axis=1) for model_name, val in metrics_train_mean.items()})

In [14]:
# select values to plot (R2 here)
r2_mets = pd.DataFrame(ds.loc['R2',:].rename(model_name) for model_name, ds in metrics_test_mean.items())

In [15]:
r2_mets = r2_mets.sort_values(axis=1, by=r'$CNN_{ft}$', ascending=False)
r2_mets = r2_mets.rename(dict(zip(column_names, graph_names)), axis=1)
r2_mets = r2_mets.rename(rename_models, axis=0)

In [16]:
r2_mets

,chla,chlc12,fuco,peri,diad,chlb,allo,but,chlide,diato,hex,zea
$CNN_{ft}$,0.770875,0.706369,0.701201,0.685386,0.677783,0.645425,0.623559,0.539100,0.501054,0.496587,0.371794,0.333607
$CNN_{sat}$,0.704584,0.633023,0.657017,0.561831,0.620584,0.560095,0.487923,0.270706,0.429194,0.472730,0.112703,0.230268


In [17]:
r2_mets

,chla,chlc12,fuco,peri,diad,chlb,allo,but,chlide,diato,hex,zea
$CNN_{ft}$,0.770875,0.706369,0.701201,0.685386,0.677783,0.645425,0.623559,0.539100,0.501054,0.496587,0.371794,0.333607
$CNN_{sat}$,0.704584,0.633023,0.657017,0.561831,0.620584,0.560095,0.487923,0.270706,0.429194,0.472730,0.112703,0.230268


In [18]:
colors = ['#1E5799', '#00A0B0']

radial_chart(r2_mets.values, r2_mets.columns, r2_mets.index, colors=colors, markers=markers, linestyles=linestyles, title='', save=None)

In [15]:
metrics_test_mean_mean

,$CNN_{ft}$,$CNN_{sat}$
MAE,0.111245,0.126751
MAPE,0.748279,0.949031
ME,-0.055999,-0.061957
MPE,0.349705,0.539231
MSE,0.123596,0.154226
R2,0.587728,0.478388


In [16]:
#  5 vs 11:

experiment_name_cnn = 'multi'
experiment_name_cnn_ast = 'OLCI'

path_experiments_cnn =  Path(f'../../experiments/{experiment_name_cnn}')
paths_metrics_cnn = [p / 'metrics' for p in path_experiments_cnn.iterdir()]

path_experiments_cnn_ast =  Path(f'../../experiments/{experiment_name_cnn_ast}')
paths_metrics_cnn_ast = [p / 'metrics' for p in path_experiments_cnn_ast.iterdir()]

# Load metrics:
metrics_test = {
    r'$CNN$': { p.parent.name :pd.read_csv(p/ (cnn.name + '_test.csv'), index_col=0) for p in paths_metrics_cnn_ast},
    r'$CNN^*$': { p.parent.name :pd.read_csv(p/ (cnn.name + '_test.csv'), index_col=0) for p in paths_metrics_cnn},
               }
metrics_train = {r'$CNN$': { p.parent.name :pd.read_csv(p/ (cnn.name + '_train.csv'), index_col=0) for p in paths_metrics_cnn_ast},
                 r'$CNN^*$': { p.parent.name :pd.read_csv(p/ (cnn.name + '_train.csv'), index_col=0) for p in paths_metrics_cnn},
               }        

In [17]:
metrics_test_mean = {model_name:  pd.concat(mets.values()).groupby(level=0).mean() for model_name, mets in metrics_test.items()}
metrics_train_mean = {model_name:  pd.concat(mets.values()).groupby(level=0).mean() for model_name, mets in metrics_train.items()}
metrics_test_mean_mean = pd.DataFrame({model_name: val.mean(axis=1) for model_name, val in metrics_test_mean.items()})
metrics_train_mean_mean = pd.DataFrame({model_name: val.mean(axis=1) for model_name, val in metrics_train_mean.items()})

# select values to plot (R2 here)
r2_mets = pd.DataFrame(ds.loc['R2',:].rename(model_name) for model_name, ds in metrics_test_mean.items())

r2_mets = r2_mets.sort_values(axis=1, by=r'$CNN$', ascending=False)
r2_mets = r2_mets.rename(dict(zip(column_names, graph_names)), axis=1)
r2_mets = r2_mets.rename(rename_models, axis=0)

r2_mets

,chlc12,chla,fuco,betac,diadino,peri,chlb,diato,chlide,19'hxfuco,allo,zea,19'btfuco
$CNN$,0.929390,0.916206,0.908330,0.907215,0.891906,0.838785,0.783394,0.782221,0.777933,0.681189,0.608167,0.543872,0.518867
$CNN^*$,0.925064,0.907958,0.908219,0.893581,0.888608,0.850047,0.764915,0.752506,0.776724,0.686888,0.603514,0.541505,0.499856


In [18]:
metrics_test_mean_mean

,$CNN$,$CNN^*$
MAE,0.043647,0.044507
MAPE,0.545656,0.514315
ME,-0.013982,-0.012873
MPE,0.259788,0.217597
MSE,0.063260,0.065166
R2,0.775960,0.769183


In [19]:
colors = ['#1E5799', '#00A0B0']

radial_chart(r2_mets.values, r2_mets.columns, r2_mets.index, colors=colors, markers=markers, linestyles=linestyles, title='', save=None)

In [19]:
#  fine tunning model vs scratch 5:

experiment_name_cnn = 'multi_sat_scratch'
experiment_name_cnn_ast = 'multi_sat_ft'

path_experiments_cnn =  Path(f'../../experiments/{experiment_name_cnn}')
paths_metrics_cnn = [p / 'metrics' for p in path_experiments_cnn.iterdir()]

path_experiments_cnn_ast =  Path(f'../../experiments/{experiment_name_cnn_ast}')
paths_metrics_cnn_ast = [p / 'metrics' for p in path_experiments_cnn_ast.iterdir()]

# Load metrics:
metrics_test = {
    r'$CNN^*_{ft}$': { p.parent.name :pd.read_csv(p/ (cnn.name + '_test.csv'), index_col=0) for p in paths_metrics_cnn_ast},
    r'$CNN^*_{sat}$': { p.parent.name :pd.read_csv(p/ (cnn.name + '_test.csv'), index_col=0) for p in paths_metrics_cnn},
               }
metrics_train = {r'$CNN^*_{ft}$': { p.parent.name :pd.read_csv(p/ (cnn.name + '_train.csv'), index_col=0) for p in paths_metrics_cnn_ast},
                 r'$CNN^*_{sat}$': { p.parent.name :pd.read_csv(p/ (cnn.name + '_train.csv'), index_col=0) for p in paths_metrics_cnn},
               } 

metrics_test_mean = {model_name:  pd.concat(mets.values()).groupby(level=0).mean() for model_name, mets in metrics_test.items()}
metrics_train_mean = {model_name:  pd.concat(mets.values()).groupby(level=0).mean() for model_name, mets in metrics_train.items()}
metrics_test_mean_mean = pd.DataFrame({model_name: val.mean(axis=1) for model_name, val in metrics_test_mean.items()})
metrics_train_mean_mean = pd.DataFrame({model_name: val.mean(axis=1) for model_name, val in metrics_train_mean.items()})

# select values to plot (R2 here)
r2_mets = pd.DataFrame(ds.loc['R2',:].rename(model_name) for model_name, ds in metrics_test_mean.items())

r2_mets = r2_mets.sort_values(axis=1, by=r'$CNN^*_{ft}$', ascending=False)
r2_mets = r2_mets.rename(dict(zip(column_names, graph_names)), axis=1)
r2_mets = r2_mets.rename(rename_models, axis=0)

colors = ['#1E5799', '#00A0B0']

radial_chart(r2_mets.values, r2_mets.columns, r2_mets.index, colors=colors, markers=markers, linestyles=linestyles, title='', save=None)

In [5]:
data = pd.read_csv("../../data/datasets/hplc_world/hplc_multi.csv", low_memory=False)

In [15]:
data["chlide_a[mg*m^3]"].plot.hist(bins=100)

<Axes: ylabel='Frequency'>